# Analysis Notebook for Radio Star scans using BigDish

dsheen 2025-10-02

The following is a notebook for analysis of radio star scans measured using scan_radio_star.py. Over the course of the notebook we will import the log files, perform rfi-robust processing of the measured data, and use the result to estimate calibrator curves and the telescope sensitivity over the full protected band.

The results of this analysis for a known reference star can then be used to perform wideband calibration of scans of unknown sources.

## 1. Manual Generation of Radio Star Flux Curves

In [1]:
import numpy as np
import astropy.units as u
import astropy.constants as const
import matplotlib.pyplot as plt

### Create a Nice Dictionary of Radio Stars we have Good Published Measurements for
use sources from

"An Accurate Flux Density Scale from 50MHz to 50GHz"

R. A. Perley and B. J. Butler National Radio Astronomy Observatory, P.O. Box O, Socorro, NM 87801, USA; RPerley@nrao.edu, BButler@nrao.edu

https://iopscience.iop.org/article/10.3847/1538-4365/aa6df9/pdf

so that we know we have good flux models for them. Only use relatively bright ones that are easy to find. Also save flux coefficents for them from table 6 so we can use this equation

$log(S)=a_0 +a_1log(f)+a_2[log(f)]^2 +a_3[log(f)]^3+ $.... for f in GHz and S in Jy

In [2]:
radio_stars = {}

radio_stars['3C48'] = {'coords':(24.42208192967, 33.15974455245), 
                       'coefficients' : np.array([1.3253, -0.7553,-0.1914,0.0498])}

radio_stars['FornaxA'] = {'coords':(50.6741208, -37.2082), 
                       'coefficients' : np.array([2.2175, -0.6606])}

radio_stars['3C123'] = {'coords':(69.26823064375, 29.67050498975), 
                       'coefficients' : np.array([1.8017, -0.7884, -0.1035, -0.0248, 0.0090])}

radio_stars['3C138'] = {'coords':(80.29119151075, 16.63945876275), 
                       'coefficients' : np.array([1.0088, -0.4981, -0.1552, -0.0102, 0.0223])}

radio_stars['PictorA'] = {'coords':(79.95717888494, -45.77884796187), 
                       'coefficients' : np.array([1.9380, -0.7470, -0.0739])}

radio_stars['TaurusA'] = {'coords':(83.6324, 22.0174), 
                       'coefficients' : np.array([2.9516, -0.2173, -0.0473, -0.0674]),
                    'beamwidth_correction' : 4.6}

radio_stars['3C147'] = {'coords':(85.65057457125, 49.852009364972), 
                       'coefficients' : np.array([1.4516, -0.6961, -0.2007, 0.0640, -0.0464, 0.0289])}

radio_stars['3C196'] = {'coords':(123.40023382088, 48.21740137945), 
                       'coefficients' : np.array([1.2872, -0.8530, -0.1534, -0.0200, 0.0201])}

radio_stars['HydraA'] = {'coords':(139.52361869175, -12.095501689528), 
                       'coefficients' : np.array([1.7795, -0.9176, -0.0843, -0.0139, 0.0295])}

radio_stars['VirgoA'] = {'coords':(187.70593076725, 12.391123246083), 
                       'coefficients' : np.array([2.4466, -0.8116, -0.0483]),
                         'beamwidth_correction' : 2.5}

radio_stars['3C286'] = {'coords':(202.78453479432, 30.50915558099), 
                       'coefficients' : np.array([1.2481, -0.4507, -0.1798, 0.0357])}

radio_stars['3C295'] = {'coords':(212.836, 52.2025), 
                       'coefficients' : np.array([1.4701, -0.7658, -0.2780, -0.0347, 0.0399])}

radio_stars['HerculesA'] = {'coords':(252.78328611990997, 4.99320753652), 
                       'coefficients' : np.array([1.8298, -1.0247, -0.0951])}

radio_stars['3C353'] = {'coords':(260.11736689063, -0.9796062372), 
                       'coefficients' : np.array([1.8627, -0.6938, -0.0998, -0.0732])}

radio_stars['3C380'] = {'coords':(277.38242072207, 48.74615560247), 
                       'coefficients' : np.array([1.2320, -0.7909, 0.0947, 0.0976, -0.1794, -0.1566])}

radio_stars['CygnusA'] = {'coords':(299.868152368208, 40.733915897917), 
                       'coefficients' : np.array([3.3498, -1.0022, -0.2246, 0.0227, 0.0425]),
                         'beamwidth_correction' : 2.5}

radio_stars['3C444'] = {'coords':(333.6073, -17.0267472), 
                       'coefficients' : np.array([1.1064, -1.0052, -0.0750, -0.0767])}

radio_stars['CassiopeiaA'] = {'coords':(350.8584, 58.8113), 
                        'coefficients' : np.array([3.3584, -0.7518, -0.0347, -0.0705]),
                        'beamwidth_correction' : 4.6}

radio_stars['Orion'] = {'coords':(83.8201, -5.3876), 
                       'coefficients' : np.array([2.705, -0.204]),
                       'beamwidth_correction' : 4.6}

In [3]:
def get_flux(star,f):
    '''
    star: str, name of star correcponding to dictinary entry key
    f: frequency or frequencies at which to compute flux
    '''
    
    a = radio_stars[star]['coefficients']
    logfreq = np.log10(f.to(u.GHz).value)
    
    
    logS = np.zeros_like(logfreq)
    for i in range(len(a)):
        logS += a[i]*np.power(logfreq,i)

    S = np.power(10,logS)*u.Jy
    return S

### Antenna Parameters and Corrections

from the corrections in RECOMMENDATION ITU-R S.733-2

$\frac{G}{T} = \frac{8\pi k (r-1)}{\lambda^2 \Phi(f)}$

where $r = \frac{P_n + P_{st}}{P_n} = \frac{T_{sys} + T_{st}}{T_{sys}}$ and $\Phi(f)$ is the flux in $Wm^{-2}Hz^{-1}$ 


corrrection factor C2 is included for the angular extent of radio stars vs the beam width in equation 3

$(G/ T)_c = G/T + C1 + C2 + C3$

where C3 is only applicable to Cas A, C1 is a correction for atmospheric loss (which we shall ignore because I typically count that as part of G/T) and so C2 is the only thing that matters


$C2 \approx -10log_{10}\left[\frac{ABS \left( 1-e^{-\chi^2} \right)}{\chi^2}\right]$ in dB

where $\chi_{CygA} \approx \frac{2.5}{1.2012 \theta_{3dB} \times 60}$

and $\chi_{most other things} \approx \frac{4.6}{1.2012 \theta_{3dB} \times 60}$

In [4]:
def get_C2(star,Theta):
    '''
    star is name of star in dictionary
    Theta is half power beamwidth in degrees
    freq is frequency
    
    '''
    Chi = radio_stars[star]['beamwidth_correction']/(1.2012*Theta.to(u.degree).value*60)
    C2 = -10*np.log10(np.abs(1-np.exp(-Chi**2))/Chi**2)
    return C2

## 2. Import Log Data from Scan of Radio Star

Before we can do anything with the actual RF samples, we need to know where the telescope was pointed at what time and what the calibrator states were. for this we need to ingest and process the log file since we do not yet have a mechanism for adding this to the drf metadata during the recording.